In [8]:
import math
import re
import string
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score
from sklearn.model_selection import train_test_split

DATA_PATH = Path("/content/spam.csv")

STOP_WORDS: frozenset[str] = frozenset(
    "a an the is are was were be been being have has had do does did "
    "will would could should may might shall can need dare ought used "
    "i me my we our you your he she it his her its they them their "
    "this that these those and but or nor so yet for of in on at to "
    "by with from up out about into through during before after above "
    "below between each few more most other some such no than then "
    "too very just here there what which who whom when where why how "
    "all both each few more most other some such only own same so "
    "than too s t ll re ve d m".split()
)


In [10]:
# ---------------------------------------------------------------------------
# Step 1 - Data loading
# ---------------------------------------------------------------------------



def load_dataset(path: Path) -> tuple[list[str], list[int]]:
    """Load the SMS spam CSV and return (texts, labels) where label 1 = spam."""
    df = pd.read_csv(path, encoding="latin-1", usecols=[0, 1])
    df.columns = ["label", "text"]
    texts = df["text"].astype(str).tolist()
    labels = (df["label"].str.lower() == "spam").astype(int).tolist()
    return texts, labels




In [13]:
# ---------------------------------------------------------------------------
# Step 2 - Preprocessing
# ---------------------------------------------------------------------------

def preprocess(text: str, remove_stopwords: bool = True) -> list[str]:
    """
    Normalise and tokenise a single SMS message.

    1. Lowercase.
    2. Remove punctuation and digits (keep alphabetic only).
    3. Split on whitespace.
    4. Optionally filter stop words.
    5. Filter single-character tokens.
    """
    # 1. Lowercase text
    text = text.lower()

    # 2. Remove punctuation and digits
    text = re.sub(r'[^a-z\s]', '', text) # Keep only lowercase alphabet and whitespace

    # 3. Split into tokens
    tokens: list[str] = text.split()

    # 4. If remove_stopwords is True, remove tokens from STOP_WORDS
    if remove_stopwords:
        tokens = [token for token in tokens if token not in STOP_WORDS]

    # 5. Remove single-character tokens
    tokens = [token for token in tokens if len(token) > 1]
    return tokens


In [14]:
# ---------------------------------------------------------------------------
# Step 3 - Vocabulary
# ---------------------------------------------------------------------------

def build_vocabulary(corpus: list[str], max_vocab: int = 3000) -> dict[str, int]:
    """
    Build a token -> index mapping from the training corpus.

    1. Preprocess every document.
    2. Count token frequencies.
    3. Keep the top max_vocab tokens.
    4. Return {token: index} ordered by descending frequency.
    """
    all_tokens = []
    for doc in corpus:
        all_tokens.extend(preprocess(doc))

    # 2. Count token frequencies.
    token_counts = Counter(all_tokens)

    # 3. Keep the top max_vocab tokens.
    # Using most_common to get tokens sorted by frequency. +1 for unknown token, if applicable
    # Here we are just taking top max_vocab, assuming vocab starts from index 0.
    top_tokens = [token for token, count in token_counts.most_common(max_vocab)]

    # 4. Return {token: index} ordered by descending frequency.
    vocab: dict[str, int] = {token: i for i, token in enumerate(top_tokens)}

    return vocab


In [15]:
# ---------------------------------------------------------------------------
# Step 4 - Bag-of-Words (Count Vectorizer)
# ---------------------------------------------------------------------------

def count_vectorize(texts: list[str], vocab: dict[str, int]) -> np.ndarray:
    """
    Convert texts to a count (Bag-of-Words) matrix.

    Returns numpy array of shape (n_docs, vocab_size), dtype int32.
    """
    n_docs = len(texts)
    vocab_size = len(vocab)
    matrix = np.zeros((n_docs, vocab_size), dtype=np.int32)

    # for each document:
    # - preprocess document
    # - increment matrix[i, vocab[token]] for each token found in vocab
    for i, text in enumerate(texts):
        tokens = preprocess(text, remove_stopwords=True) # Assuming stop words should be removed for BOW
        for token in tokens:
            if token in vocab:
                matrix[i, vocab[token]] += 1

    return matrix


In [16]:
# ---------------------------------------------------------------------------
# Step 5 - TF-IDF
# ---------------------------------------------------------------------------

def compute_idf(corpus_tokens: list[list[str]], vocab: dict[str, int]) -> np.ndarray:
    """
    Compute the IDF vector for all vocabulary tokens.

    Formula (smoothed):
        IDF(t) = log((1 + N) / (1 + df(t))) + 1
    """
    n_docs = len(corpus_tokens)
    vocab_size = len(vocab)
    df = np.zeros(vocab_size, dtype=np.float64)

    # compute document frequency per token
    for doc_tokens in corpus_tokens:
        unique_tokens = set(doc_tokens)
        for token in unique_tokens:
            if token in vocab:
                df[vocab[token]] += 1

    # compute and return smoothed IDF vector
    idf = np.log((1 + n_docs) / (1 + df)) + 1
    return idf


def tfidf_vectorize(
    texts: list[str],
    vocab: dict[str, int],
    idf: np.ndarray,
) -> np.ndarray:
    """
    Convert texts to a TF-IDF matrix.

        TF(t, d)     = count(t, d) / len(d)
        TF-IDF(t, d) = TF(t, d) * IDF(t)

    IDF is passed in (pre-computed on training data) to prevent leakage.

    Returns numpy array of shape (n_docs, vocab_size), dtype float64.
    """
    n_docs = len(texts)
    vocab_size = len(vocab)
    matrix = np.zeros((n_docs, vocab_size), dtype=np.float64)

    for i, text in enumerate(texts):
        processed_tokens = preprocess(text, remove_stopwords=True)
        token_counts = Counter(processed_tokens)
        doc_length = len(processed_tokens)

        if doc_length == 0:  # Avoid division by zero for empty documents
            continue

        for token, count in token_counts.items():
            if token in vocab:
                tf = count / doc_length
                token_idx = vocab[token]
                matrix[i, token_idx] = tf * idf[token_idx]
    pass
    return matrix


In [17]:
# ---------------------------------------------------------------------------
# Step 6 - Custom hand-crafted features
# ---------------------------------------------------------------------------

_SPAM_WORDS = frozenset(
    "free win winner won cash prize claim urgent call txt text reply "
    "guaranteed offer discount limited mobile ringtone download bonus "
    "selected congratulations awarded voucher reward collect".split()
)


def extract_custom_features(text: str) -> list[float]:
    """
    Hand-crafted feature vector for one SMS message (9 features).

    Suggested features:
    1. Message length (characters)
    2. Number of tokens
    3. Number of digits
    4. Number of uppercase characters
    5. Uppercase character ratio
    6. Number of punctuation marks (!, ?, .)
    7. Number of currency symbols ($, GBP, EUR)
    8. Count of spam-indicator words (lowercased)
    9. Type-token ratio (unique tokens / total tokens)
    """
    features = []

    # 1. Message length (characters)
    features.append(float(len(text)))

    # Preprocess text to get tokens (without stopword removal for some features)
    all_tokens = preprocess(text, remove_stopwords=False)
    # Preprocess text to get tokens (with stopword removal for type-token ratio)
    processed_tokens = preprocess(text, remove_stopwords=True)

    # 2. Number of tokens
    num_tokens = len(all_tokens)
    features.append(float(num_tokens))

    # 3. Number of digits
    num_digits = sum(c.isdigit() for c in text)
    features.append(float(num_digits))

    # 4. Number of uppercase characters
    num_uppercase = sum(c.isupper() for c in text)
    features.append(float(num_uppercase))

    # 5. Uppercase character ratio
    uppercase_ratio = num_uppercase / len(text) if len(text) > 0 else 0.0
    features.append(uppercase_ratio)

    # 6. Number of punctuation marks (!, ?, .)
    num_punctuation = sum(1 for c in text if c in string.punctuation)
    features.append(float(num_punctuation))

    # 7. Number of currency symbols ($, GBP, EUR)
    currency_symbols_count = len(re.findall(r'\$|GBP|EUR', text, re.IGNORECASE))
    features.append(float(currency_symbols_count))

    # 8. Count of spam-indicator words (lowercased)
    spam_word_count = sum(1 for token in all_tokens if token in _SPAM_WORDS)
    features.append(float(spam_word_count))

    # 9. Type-token ratio (unique tokens / total tokens)
    unique_tokens = set(processed_tokens)
    type_token_ratio = len(unique_tokens) / len(processed_tokens) if len(processed_tokens) > 0 else 0.0
    features.append(type_token_ratio)

    return features


In [23]:
# ---------------------------------------------------------------------------
# Step 7 - Evaluation helper
# ---------------------------------------------------------------------------

def evaluate(
    X_train: np.ndarray,
    X_test: np.ndarray,
    y_train: list[int],
    y_test: list[int],
    label: str,
) -> None:
    """Train logistic regression and print a classification report."""
    print(f"\n--- Evaluating: {label} ---")
    # Train LogisticRegression(max_iter=1000, random_state=42)
    model = LogisticRegression(max_iter=1000, random_state=42)
    model.fit(X_train, y_train)

    # Predict on X_test
    y_pred = model.predict(X_test)

    # Compute macro F1
    macro_f1 = f1_score(y_test, y_pred, average='macro')
    print(f"Macro F1-score: {macro_f1:.2f}")

    # Print section header and classification_report
    print(classification_report(y_test, y_pred, target_names=['ham', 'spam']))


In [22]:
# ---------------------------------------------------------------------------
# Main
# ---------------------------------------------------------------------------

def main() -> None:
    print("Loading dataset...")
    texts, labels = load_dataset(DATA_PATH)

    print(f"  Total samples : {len(texts)}")
    print(f"  Spam          : {sum(labels)}  ({sum(labels)/len(labels):.1%})")
    print(f"  Ham           : {len(labels)-sum(labels)}")

    X_train_raw, X_test_raw, y_train, y_test = train_test_split(
        texts, labels, test_size=0.5, random_state=42, stratify=labels
    )

    # build vocabulary from X_train_raw only (avoid leakage)
    vocab: dict[str, int] = build_vocabulary(X_train_raw)

    # --- Bag-of-Words ---
    print("\n--- Bag-of-Words ---")
    X_train_bow = count_vectorize(X_train_raw, vocab)
    X_test_bow = count_vectorize(X_test_raw, vocab)
    evaluate(X_train_bow, X_test_bow, y_train, y_test, "Bag-of-Words")

    # --- TF-IDF ---
    print("\n--- TF-IDF ---")
    # preprocess X_train_raw -> tokens
    X_train_tokens = [preprocess(text) for text in X_train_raw]
    # compute idf on training tokens only
    idf = compute_idf(X_train_tokens, vocab)
    # vectorize train/test with tfidf_vectorize and evaluate
    X_train_tfidf = tfidf_vectorize(X_train_raw, vocab, idf)
    X_test_tfidf = tfidf_vectorize(X_test_raw, vocab, idf)
    evaluate(X_train_tfidf, X_test_tfidf, y_train, y_test, "TF-IDF")

    # --- Custom features ---
    print("\n--- Custom features ---")
    # build numpy feature matrices from extract_custom_features
    X_train_custom = np.array([extract_custom_features(text) for text in X_train_raw])
    X_test_custom = np.array([extract_custom_features(text) for text in X_test_raw])
    # evaluate custom features
    evaluate(X_train_custom, X_test_custom, y_train, y_test, "Custom Features")

    # --- Custom approach (optional) ---
    # TODO: add your own representation idea and evaluate it
    # (e.g., use scikit-learn tf-idf implementation and try to find the best hyperparameter values)
    # (run script using machine learning models)


if __name__ == "__main__":
    main()


Loading dataset...
  Total samples : 5572
  Spam          : 747  (13.4%)
  Ham           : 4825

--- Bag-of-Words ---

--- Evaluating: Bag-of-Words ---
Macro F1-score: 0.93
              precision    recall  f1-score   support

         ham       0.97      1.00      0.98      2413
        spam       0.99      0.79      0.88       373

    accuracy                           0.97      2786
   macro avg       0.98      0.90      0.93      2786
weighted avg       0.97      0.97      0.97      2786


--- TF-IDF ---

--- Evaluating: TF-IDF ---
Macro F1-score: 0.91
              precision    recall  f1-score   support

         ham       0.96      1.00      0.98      2413
        spam       0.96      0.75      0.85       373

    accuracy                           0.96      2786
   macro avg       0.96      0.87      0.91      2786
weighted avg       0.96      0.96      0.96      2786


--- Custom features ---

--- Evaluating: Custom Features ---
Macro F1-score: 0.95
              precision  

Among the around 5600 considered SMS, 747 (13.4%) are spam ones. All considered feature representation methods (Bag of words, TF-IDF and custom one) get good results, having a F1 score above 0.9. More specifically, the latter one gets the best results, with a F1 score of 0.95.